In [ ]:
# ========== 导入：把后面 Gradio 多模型文档字符串工具要用到的库搬进来 ==========

# 导入标准库 os：读环境变量等（本笔记本主要用 Colab userdata，os 备用）
import os
# 导入 black：把模型生成的 Python 代码按 PEP 8 重新格式化（format_str）
import black
# 从 openai 导入 OpenAI 客户端：统一用 Chat Completions 风格调云端模型
from openai import OpenAI
# 从 google.colab 导入 userdata：从 Colab Secrets 安全读取 API Key（不写进笔记本）
from google.colab import userdata
# 导入 gradio：快速搭 Web UI（Blocks / Button / Code 等）
import gradio as gr


In [ ]:
# 安装 black：Colab 环境默认可能没有，先 pip 再导入（与上面 import black 配套）
!pip install black


In [7]:
# ========== API Key：从 Colab Secrets 读出，供 OpenAI / Anthropic 客户端使用 ==========

# 从 Colab Secrets 读取 OpenAI 密钥（名字必须与 Secrets 里配置的一致）
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
# 从 Colab Secrets 读取 Anthropic 密钥（后面经 OpenAI 兼容 base_url 调用）
ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")

# 若 OpenAI 密钥缺失：立刻抛错，避免后续请求静默失败
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found in Colab Secrets.")

# 若 Anthropic 密钥缺失：同样硬失败
if not ANTHROPIC_API_KEY:
    raise ValueError("ANTHROPIC_API_KEY not found in Colab Secrets.")

# 两个密钥都到手：打印确认（不要 print 密钥本身）
print("API Keys Loaded Successfully")


✅ API Keys Loaded Successfully


In [8]:
# ========== 客户端初始化：一个官方 OpenAI，一个走 Anthropic 兼容端点 ==========

# 创建 OpenAI 官方客户端：后续 provider="openai" 时用它
openai_client = OpenAI(api_key=OPENAI_API_KEY)

# 创建「OpenAI SDK + Anthropic base_url」客户端：用同一套 chat.completions API 调 Claude
anthropic_client = OpenAI(
    api_key=ANTHROPIC_API_KEY,
    base_url="https://api.anthropic.com/v1"
)

# 初始化成功提示（不含密钥）
print("Clients initialized successfully")


✅ Clients initialized successfully


In [ ]:
# ========== 模型名常量：high/low 两档，OpenAI 与 Claude 各一对 ==========

# OpenAI 高档：更强的 gpt-4o（贵、质量高）
OPENAI_HIGH = "gpt-4o"
# OpenAI 低档：更快更便宜的 gpt-4o-mini
OPENAI_LOW = "gpt-4o-mini"

# Claude 高档：opus（强推理，成本高）
CLAUDE_HIGH = "claude-3-opus-20240229"
# Claude 低档：haiku（快、便宜）
CLAUDE_LOW = "claude-3-haiku-20240307"


In [10]:
# ========== 路由：按 provider + tier 选出「客户端对象」和「模型 id」 ==========

def get_client_and_model(provider="openai", tier="low"):
    # provider / tier 默认 openai + low，UI 下拉会覆盖这两个参数

    # OpenAI 分支：按档位在 HIGH / LOW 常量里选模型
    if provider == "openai":
        model = OPENAI_HIGH if tier == "high" else OPENAI_LOW
        return openai_client, model

    # Claude 分支：用 anthropic_client + 对应 Claude 模型 id
    elif provider == "claude":
        model = CLAUDE_HIGH if tier == "high" else CLAUDE_LOW
        return anthropic_client, model

    # 未知 provider：拒绝，避免静默落到错误后端
    else:
        raise ValueError("Unsupported provider.")


In [ ]:
# ========== System Prompt：教模型「只改注释/文档、不改逻辑」，可选追加 pytest ==========

def build_system_prompt(include_tests=False):
    # include_tests=True 时，在 base 规则后再追加「生成单元测试」条款

    # 基础 system 指令：要求返回纯可执行 Python（不要 markdown 围栏）
    base_prompt = """
You are a senior Python engineer.

Your task:

1. Add professional docstrings.
2. Add meaningful inline comments.
3. Do NOT change logic.
4. Do NOT wrap in markdown.
5. Return ONLY raw executable Python code.

Docstrings must include:
- Description
- Args
- Returns
"""

    # 勾选「生成测试」时：追加第 6 条，要求在代码后输出 pytest
    if include_tests:
        base_prompt += """

6. After the updated code, generate pytest unit tests.
   - Cover normal and edge cases.
   - Use professional structure.
"""

    # 返回拼好的完整 system prompt 字符串
    return base_prompt


In [ ]:
# ========== PEP 8 格式化：用 black 美化模型输出；失败则原样返回 ==========

def format_code_pep8(code):
    try:
        # black.format_str：按 FileMode 默认规则格式化字符串里的 Python
        return black.format_str(code, mode=black.FileMode())
    except Exception:
        # 模型偶发返回非法语法时 black 会抛错：吞掉异常，保证 UI 仍能展示原文
        return code


In [ ]:
# ========== 非流式生成：一次拿齐模型回复 + token/费用元数据 ==========

def generate_docstrings(code, provider="openai", tier="low", include_tests=False):
    # code：用户粘贴的 Python；其余参数与 UI / get_client_and_model 对齐

    # 解析出要用的 client 与 model id
    client, model = get_client_and_model(provider, tier)
    # 按是否要测试组装 system prompt
    system_prompt = build_system_prompt(include_tests)

    # 非流式 Chat Completions：temperature=0 降低随机性，便于代码改写稳定
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": code}
        ],
        temperature=0
    )

    # 取出助手文本，再用 black 做 PEP 8 格式化
    content = response.choices[0].message.content
    content = format_code_pep8(content)

    # usage：API 返回的 token 统计对象
    usage = response.usage
    total_tokens = usage.total_tokens

    # 粗算费用：每 1k token 约 $0.005（示例单价，非账单真值）
    cost_per_1k_tokens = 0.005
    estimated_cost = (total_tokens / 1000) * cost_per_1k_tokens

    # 拼给 UI「Token Usage & Cost」文本框的元数据
    metadata = f"""
Prompt Tokens: {usage.prompt_tokens}
Completion Tokens: {usage.completion_tokens}
Total Tokens: {usage.total_tokens}
Estimated Cost: ${estimated_cost:.6f}
"""

    # 返回：(格式化后的代码, 用量说明)
    return content, metadata


In [29]:
# ========== 流式生成：边生成边 yield，结束后再 black + 补一次用量查询 ==========

def generate_docstrings_stream(code, provider="openai", tier="low", include_tests=False):
    # 与非流式同签名；用 generator（yield）把中间结果推给 Gradio

    # 选客户端与模型
    client, model = get_client_and_model(provider, tier)
    # 组装 system prompt
    system_prompt = build_system_prompt(include_tests)

    # stream=True：服务端按 chunk 推送 delta.content
    stream = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": code}
        ],
        temperature=0,
        stream=True
    )

    # 累积完整回复文本
    full_response = ""

    # 逐块拼接；第二个返回值先留空，等流结束后再填 metadata
    for chunk in stream:
        if chunk.choices[0].delta.content:
            content = chunk.choices[0].delta.content
            full_response += content
            yield full_response, ""   # live stream

    # ---------- 流结束后的后处理 ----------

    # 对完整文本做 black 格式化
    formatted = format_code_pep8(full_response)

    # 流式响应通常不带完整 usage：再发一次非流式同请求，只为拿 token 统计
    final_response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": code}
        ],
        temperature=0
    )

    # 从第二次响应读 usage，并粗算费用
    usage = final_response.usage
    total_tokens = usage.total_tokens
    cost_per_1k_tokens = 0.005
    estimated_cost = (total_tokens / 1000) * cost_per_1k_tokens

    # 最终 metadata（与非流式字段一致）
    metadata = f"""
Prompt Tokens: {usage.prompt_tokens}
Completion Tokens: {usage.completion_tokens}
Total Tokens: {usage.total_tokens}
Estimated Cost: ${estimated_cost:.6f}
"""

    # 最后一次 yield：格式化代码 + 完整用量
    yield formatted, metadata


In [39]:
# ========== 文件上传：把 .py 字节读成 UTF-8 字符串填进代码框 ==========

def load_file(file):
    # Gradio File 组件可能给 None（用户未选文件）
    if file is None:
        return ""
    # 读文件对象字节并按 utf-8 解码为源码文本
    return file.read().decode("utf-8")


In [40]:
# ========== 复制占位：原样返回代码（供按钮/剪贴板扩展用） ==========

def copy_output(code):
    # 不做变换，方便后续接 clipboard 逻辑时保持接口稳定
    return code


In [ ]:
# ========== 另一版 generate_docstrings：把 metadata 拼进代码字符串末尾返回 ==========
# 注意：与 cell 8 同名函数，后执行的定义会覆盖前者；本格返回单字符串而非 (code, metadata)

def generate_docstrings(code, provider="openai", tier="low", include_tests=False):
    # 同样按 provider/tier 选客户端与模型
    client, model = get_client_and_model(provider, tier)
    system_prompt = build_system_prompt(include_tests)

    # 非流式调用；temperature=0 求稳定输出
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": code}
        ],
        temperature=0
    )

    # 取出模型原文（本格不做 black）
    content = response.choices[0].message.content

    # ---------- Token 用量 ----------
    usage = response.usage
    prompt_tokens = usage.prompt_tokens
    completion_tokens = usage.completion_tokens
    total_tokens = usage.total_tokens

    # ---------- 粗略成本估算（示例单价，可按模型调整） ----------
    cost_per_1k_tokens = 0.005  # Example estimate
    estimated_cost = (total_tokens / 1000) * cost_per_1k_tokens

    # 把用量以「注释块」形式接到代码后面（与 cell 8 的双返回值设计不同）
    metadata = f"\n\n# --- METADATA ---\n# Prompt Tokens: {prompt_tokens}\n# Completion Tokens: {completion_tokens}\n# Total Tokens: {total_tokens}\n# Estimated Cost: ${estimated_cost:.6f}\n"

    # 返回：代码 + 注释形式的 metadata
    return content + metadata


In [ ]:
# ========== Gradio UI：多模型文档字符串 / 单测生成器主界面 ==========

def docstring_app(code, provider, tier, streaming, include_tests):
    # Gradio 回调：根据 streaming 开关走流式或非流式路径

    # 空输入：提示用户粘贴有效 Python（提示文案保持原样，供运行判断）
    if not code or not code.strip():
        yield "#Please paste valid Python code.", ""
        return

    # 流式：把 generator 的中间结果一路 yield 给前端
    if streaming:
        yield from generate_docstrings_stream(
            code, provider, tier, include_tests
        )
    else:
        # 非流式：一次拿到 (content, metadata) 再 yield
        content, metadata = generate_docstrings(
            code, provider, tier, include_tests
        )
        yield content, metadata

def clear_fields():
    # Clear 按钮：清空输入与输出相关字段
    return "", ""


# 用 Blocks 搭布局（比 Interface 更灵活）
with gr.Blocks() as demo:

    # 标题与简介（给终端用户看的 UI 文案）
    gr.Markdown("## AI Docstring & Unit Test Generator")
    gr.Markdown("Production-ready multi-model AI code enhancement tool.")

    # 第一行：Provider + Model Tier 两个下拉
    with gr.Row():
        provider = gr.Dropdown(
            choices=["openai", "claude"],
            value="openai",
            label="Provider"
        )
        tier = gr.Dropdown(
            choices=["low", "high"],
            value="low",
            label="Model Tier"
        )

    # 是否流式输出
    streaming = gr.Checkbox(label="Enable Streaming", value=True)
    # 是否让模型顺便生成 pytest
    include_tests = gr.Checkbox(label="Generate Unit Tests", value=False)

    # 上传 .py：选中后触发 load_file → 填入 code_input
    file_upload = gr.File(label="Upload Python File (.py)", file_types=[".py"])

    # 主输入：粘贴或由文件填充的 Python 源码
    code_input = gr.Code(label="Paste Python Code", language="python")

    # 文件变更 → 读入代码框
    file_upload.change(load_file, inputs=file_upload, outputs=code_input)

    # 生成结果：代码 + 用量文本框
    output = gr.Code(label="Generated Code", language="python")
    metadata_output = gr.Textbox(label="Token Usage & Cost", lines=4)

    # Generate / Clear 按钮行
    with gr.Row():
        generate_btn = gr.Button("Generate")
        clear_btn = gr.Button("Clear", variant="secondary")

    # Generate：调用 docstring_app，写入两个输出组件
    generate_btn.click(
        docstring_app,
        inputs=[code_input, provider, tier, streaming, include_tests],
        outputs=[output, metadata_output]
    )

    # Clear：清空输入码、输出码、用量（与 clear_fields 返回个数对应）
    clear_btn.click(
        clear_fields,
        outputs=[code_input, output, metadata_output]
    )

# 启动 Gradio；debug=True 便于在笔记本里看报错栈
demo.launch(debug=True)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e1dd5ae17d58d5778e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
